# Advanced Feature Engineering — Examples

This notebook demonstrates feature creation, transformations, mixed-type preprocessing, feature selection, and leakage-safe evaluation.

**Requirements:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`. Run cells from top to bottom.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42


## 1. Domain-inspired features
Derive values from fields available at prediction time.

In [ ]:
orders = pd.DataFrame({
    "quantity": [1, 3, 2, 5, 4, 1],
    "unit_price": [250, 80, 150, 40, 120, 600],
    "discount_rate": [0.00, 0.10, 0.05, 0.00, 0.15, 0.20],
})
orders["gross_value"] = orders["quantity"] * orders["unit_price"]
orders["net_value"] = orders["gross_value"] * (1 - orders["discount_rate"])
orders["value_per_item"] = orders["net_value"] / orders["quantity"].clip(lower=1)
display(orders)


## 2. Date and time features

In [ ]:
events = pd.DataFrame({
    "timestamp": pd.to_datetime([
        "2026-01-05 09:15:00Z", "2026-01-10 18:30:00Z",
        "2026-02-12 13:00:00Z", "2026-02-15 11:45:00Z"
    ], utc=True)
})
events["month"] = events["timestamp"].dt.month
events["day_of_week"] = events["timestamp"].dt.dayofweek
events["hour"] = events["timestamp"].dt.hour
events["is_weekend"] = (events["day_of_week"] >= 5).astype(int)
display(events)


## 3. Nonlinear transformation
`log1p` requires values greater than -1; check the domain before applying it.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
amount = rng.lognormal(mean=3.0, sigma=1.0, size=1000)
transformed = np.log1p(amount)

plt.hist(amount, bins=30)
plt.title("Original right-skewed data")
plt.show()

plt.hist(transformed, bins=30)
plt.title("After log1p transformation")
plt.show()


## 4. Binning and polynomial features

In [ ]:
values = pd.DataFrame({"value": [2, 4, 7, 9, 12, 18, 23, 31]})
values["bin"] = pd.cut(values["value"], bins=[0, 10, 20, 40],
                       labels=["low", "medium", "high"])
display(values)

poly = PolynomialFeatures(degree=2, include_bias=False)
sample = np.array([[2.0, 3.0], [4.0, 5.0], [6.0, 8.0]])
expanded = poly.fit_transform(sample)
print("Feature names:", poly.get_feature_names_out(["x1", "x2"]))
display(pd.DataFrame(expanded, columns=poly.get_feature_names_out(["x1", "x2"])))


## 5. Mixed-type preprocessing with ColumnTransformer

In [ ]:
df = pd.DataFrame({
    "age": [22, 35, np.nan, 48, 29, 41, 53, 31, 27, 45, 38, 50],
    "income": [32000, 52000, 41000, np.nan, 39000, 61000, 70000, 45000, 36000, 68000, 55000, 73000],
    "city": ["Indore", "Pune", "Indore", None, "Bhopal", "Pune", "Indore", "Bhopal", "Pune", "Indore", None, "Pune"],
    "plan": ["Basic", "Premium", "Basic", "Premium", "Basic", "Premium", "Premium", "Basic", "Basic", "Premium", "Basic", "Premium"],
})
y = (df["plan"] == "Premium").astype(int)
X = df.drop(columns="plan")

numeric = ["age", "income"]
categorical = ["city"]
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
pipeline.fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, pipeline.predict(X_test)))


This tiny synthetic dataset demonstrates the API only; it is not enough to draw reliable conclusions about customer behavior.

## 6. Feature selection inside cross-validation

In [ ]:
from functools import partial

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

mi = partial(mutual_info_classif, random_state=RANDOM_STATE)
selected_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("select", SelectKBest(score_func=mi, k=10)),
    ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])
all_features_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, estimator in [
    ("All features", all_features_pipeline),
    ("Select 10 features", selected_pipeline),
]:
    scores = cross_val_score(estimator, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: mean ROC-AUC={scores.mean():.4f}, std={scores.std():.4f}")

selected_pipeline.fit(X_train, y_train)
mask = selected_pipeline.named_steps["select"].get_support()
print("Selected features:", list(data.feature_names[mask]))
print("Held-out test ROC-AUC:",
      roc_auc_score(y_test, selected_pipeline.predict_proba(X_test)[:, 1]))


## Key takeaways

- Split data before fitting learned transformations.
- Put imputers, scalers, encoders, and feature selectors inside a pipeline.
- Use time-aware features and splits for temporal problems.
- Do not use information recorded after the target event.
- Compare against a baseline and document limitations.

Further reading: [Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html), [Pipelines](https://scikit-learn.org/stable/modules/compose.html), [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html).
